# Create Hal config and shutter files

In [ ]:
# Imports
import sys, numpy as np
sys.path.insert(0, "/path/to/merfish_pipeline/src")

from merfish_pipeline.common.config      import ExperimentConfig
from merfish_pipeline.acquisition.configs import (
    get_frame_table, get_color_sequence_name,
    create_shutter_file, create_hal_config,
)
from merfish_pipeline.acquisition.display import print_frame_table, display_xml
from merfish_pipeline.visualization       import visualize_shutter_sequence

In [ ]:
# Experiment paths 
ROOT = "/data/experiments/LT026_sample_03"

config = ExperimentConfig(
    data_dir       = f"{ROOT}/images",
    metadata_dir   = f"{ROOT}/metadata",
    analysis_dir   = f"{ROOT}/analysis",
    round_info_csv = f"{ROOT}/metadata/round_info.csv",
    positions_txt  = f"{ROOT}/metadata/positions.txt",
    microscope     = "MF3",
    hal_templates_dir = f"{ROOT}/metadata",
)
OUTPUT = "/data/experiments/LT026_sample_03/metadata"

In [ ]:
# ───── Define the imaging round ─────────────────────────────────────────
z_bead    = 0
z_pos     = np.arange(1, 25.5, 0.5)   # 1.0 µm to 25.0 µm in 0.5 µm steps
bead_seq  = [488, np.nan]
color_seq = [560, 650]
end_seq   = [np.nan, np.nan]

frame_table = get_frame_table(z_bead, bead_seq, color_seq, end_seq, z_pos)
name        = get_color_sequence_name(frame_table)
print(f"Color sequence name: {name}")
print_frame_table(frame_table)

# ───── Write shutter file ───────────────────────────────────────────────
shutter_name = f"shutter_{name}.xml"
shutter_path = f"{OUTPUT}/{shutter_name}"
create_shutter_file(frame_table, shutter_path)
display_xml(shutter_path)

# ───── Write HAL config ─────────────────────────────────────────────────
template   = config.hal_templates_dir / "hal-config-mf3-epi.xml"
hal_output = f"{OUTPUT}/hal-config-mf3-epi_{name}.xml"
create_hal_config(template, frame_table, shutter_name, hal_output)
display_xml(hal_output)

# ───── Visualise ────────────────────────────────────────────────────────
visualize_shutter_sequence(
    frame_table,
    title=f"Shutter sequence: {name}",
    save_path=f"{OUTPUT}/shutter_sequence_{name}.png",
)